# A9 Batch Inference & Aggregation (CPU)

Menjalankan full-corpus inference dan aggregation pada canonical reviews
menggunakan model TF-IDF yang terpilih setelah A8, melalui
`sipature_ml.a9.run_inference` dan `run_aggregation`.
Ikuti `docs/a9-inference-priority-report.md`.

Input: `data/processed/canonical_reviews.parquet` (notebook `02`) dan
`models/tfidf-aspect-silver-v1/` (notebook `05`).
Output: `a9/<run>-infer/` dan `a9/<run>-aggregate/` di Drive.

Output level-review bersifat **restricted** (tidak dicopy ke repo).


## Step 1 — Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")

# Input (hasil notebook 02 dan notebook 05).
REVIEWS_PATH = DRIVE_ROOT / "data" / "processed" / "canonical_reviews.parquet"
MODEL_DIR = DRIVE_ROOT / "models" / "tfidf-aspect-silver-v1"

# Output immutable (timestamp agar tidak menimpa run lama).
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M_a9-tfidf-lexical-v1")
INFERENCE_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-infer"
AGGREGATION_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-aggregate"

PROJECT_DIR = Path("/content/hackathon/ml")

print("Reviews:", REVIEWS_PATH)
print("Model dir:", MODEL_DIR)
print("Run ID:", RUN_ID)
print("Inference dir:", INFERENCE_DIR)
print("Aggregation dir:", AGGREGATION_DIR)


Reviews: /content/drive/MyDrive/SIPATURE/data/processed/canonical_reviews.parquet
Model dir: /content/drive/MyDrive/SIPATURE/models/tfidf-aspect-silver-v1
Run ID: 20260813-1713_a9-tfidf-lexical-v1
Inference dir: /content/drive/MyDrive/SIPATURE/a9/20260813-1713_a9-tfidf-lexical-v1-infer
Aggregation dir: /content/drive/MyDrive/SIPATURE/a9/20260813-1713_a9-tfidf-lexical-v1-aggregate


## Step 3 — Clone repository dari GitHub


In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 4 — Verifikasi commit terbaru (git log)


In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
97ce3e6 (HEAD -> main, origin/main, origin/HEAD) feat: add A9 inference+aggregation notebook and refresh TF-IDF model hashes
df5c9e9 docs: record notebook 07 A8 completion and mark calibration/locked-test artifacts done
d7fec30 Created using Colab


## Step 5 — Install dependencies (CPU profile)


In [5]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-dev.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.7/323.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

**RESTART WAJIB.** Setelah install, restart runtime agar numpy/sklearn lama
tidak ter-cache di memori.

1. **Runtime > Restart session**
2. Jalankan ulang **Step 1** (mount) dan **Step 2** (config)
3. Step 3–5 **tidak perlu diulang**

Lalu lanjut ke **Step 6**.


## Step 6 — Verifikasi versi package (setelah restart)


In [3]:
import joblib
import numpy
import pandas
import pyarrow
import sklearn

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

assert sklearn.__version__ == "1.7.2", (
    f"scikit-learn 1.7.2 diperlukan untuk memuat model TF-IDF, "
    f"ditemukan {sklearn.__version__}"
)
print("\nEnvironment A9 siap.")


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
Scikit-learn: 1.7.2
Joblib: 1.5.3

Environment A9 siap.


## Step 7 — Import modul sipature_ml


In [4]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 8 — Load config A9 & verifikasi kontrak model TF-IDF


In [5]:
from sipature_ml.a9 import load_tfidf_contract
from sipature_ml.config import load_config

config = load_config("a9")

print("A9 version:", config["a9_version"])
print("Aspect model:", config["models"]["aspect"]["version"])
print("Polarity model:", config["models"]["polarity"]["version"])
print("Severity status:", config["models"]["severity"]["status"])

artifact = load_tfidf_contract(MODEL_DIR, config)
print("\nModel TF-IDF terverifikasi.")
print("Jumlah aspek:", len(artifact["aspects"]))
print("Aspek:", artifact["aspects"])
print("Thresholds:", [round(t, 3) for t in artifact["thresholds"]])


A9 version: a9-tfidf-lexical-v1.0.4
Aspect model: tfidf-aspect-silver-v1
Polarity model: lexical-polarity-v1
Severity status: unavailable_no_supported_model

Model TF-IDF terverifikasi.
Jumlah aspek: 14
Aspek: ['access', 'cleanliness', 'comfort', 'crowding', 'maintenance', 'opening_hours', 'parking', 'price_transparency', 'public_facilities', 'safety', 'sanitation', 'scenery', 'staff_service', 'waste']
Thresholds: [np.float64(0.5), np.float64(0.5), np.float64(0.45), np.float64(0.5), np.float64(0.5), np.float64(0.25), np.float64(0.65), np.float64(0.5), np.float64(0.5), np.float64(0.5), np.float64(0.5), np.float64(0.45), np.float64(0.55), np.float64(0.35)]


## Step 9 — Jalankan full-corpus inference


In [6]:
from sipature_ml.a9 import run_inference

assert not INFERENCE_DIR.exists(), f"Sudah ada: {INFERENCE_DIR}"

summary = run_inference(
    reviews_path=REVIEWS_PATH,
    model_dir=MODEL_DIR,
    output_dir=INFERENCE_DIR,
)

print("A9 version:", summary["a9_version"])
print("Text reviews:", summary["text_reviews"])
print("Reviews with predictions:", summary["reviews_with_predictions"])
print("Aspect predictions:", summary["aspect_predictions"])
print("Aspect model:", summary["aspect_model"])
print("Polarity model:", summary["polarity_model"])
print("Restricted:", summary["restricted"])


A9 version: a9-tfidf-lexical-v1.0.4
Text reviews: 12234
Reviews with predictions: 5942
Aspect predictions: 9785
Aspect model: tfidf-aspect-silver-v1
Polarity model: lexical-polarity-v1
Restricted: True


## Step 10 — Jalankan aggregation (destination-aspect signals + evidence)


In [7]:
from sipature_ml.a9 import run_aggregation

assert not AGGREGATION_DIR.exists(), f"Sudah ada: {AGGREGATION_DIR}"

summary = run_aggregation(
    predictions_dir=INFERENCE_DIR,
    reviews_path=REVIEWS_PATH,
    output_dir=AGGREGATION_DIR,
)

print("A9 version:", summary["a9_version"])
print("Signals:", summary["signals"])
print("Evidence items:", summary["evidence_items"])
print("Destinations:", summary["destinations"])
print("Severity status:", summary["severity_status"])
print("Restricted:", summary["restricted"])


A9 version: a9-tfidf-lexical-v1.0.4
Signals: 1682
Evidence items: 598
Destinations: 280
Severity status: unavailable_no_supported_model
Restricted: True


## Step 11 — Verifikasi output & hash artifact


In [8]:
import json
from pathlib import Path

from sipature_ml.manifest import sha256_file

for name, directory in (("infer", INFERENCE_DIR), ("aggregate", AGGREGATION_DIR)):
    manifest = json.loads(
        (directory / "manifest.json").read_text(encoding="utf-8")
    )
    print(f"=== {name} ===")
    print("  Stage:", manifest["stage"])
    print("  A9 version:", manifest["a9_version"])
    errors = []
    for relative, expected in manifest["artifact_hashes"].items():
        path = directory / relative
        if not path.is_file():
            errors.append(f"missing: {relative}")
        elif sha256_file(path) != expected:
            errors.append(f"hash mismatch: {relative}")
    print(f"  Artifact check: {len(manifest['artifact_hashes'])} file, {len(errors)} masalah")
    assert not errors, errors

print("\nSeluruh output A9 valid terhadap manifest.")


=== infer ===
  Stage: infer
  A9 version: a9-tfidf-lexical-v1.0.4
  Artifact check: 2 file, 0 masalah
=== aggregate ===
  Stage: aggregate
  A9 version: a9-tfidf-lexical-v1.0.4
  Artifact check: 3 file, 0 masalah

Seluruh output A9 valid terhadap manifest.


## Step 12 — Run summary


In [9]:
print("RUN ID:", RUN_ID)
print("INFERENCE DIR :", INFERENCE_DIR)
print("AGGREGATION DIR:", AGGREGATION_DIR)
print("REVIEWS       :", REVIEWS_PATH)
print("MODEL DIR     :", MODEL_DIR)

print("\nREMINDER: output review-level (predictions/evidence) bersifat restricted.")
print("Lanjut ke notebook 09 untuk prioritization + export (privacy-safe).")


RUN ID: 20260813-1713_a9-tfidf-lexical-v1
INFERENCE DIR : /content/drive/MyDrive/SIPATURE/a9/20260813-1713_a9-tfidf-lexical-v1-infer
AGGREGATION DIR: /content/drive/MyDrive/SIPATURE/a9/20260813-1713_a9-tfidf-lexical-v1-aggregate
REVIEWS       : /content/drive/MyDrive/SIPATURE/data/processed/canonical_reviews.parquet
MODEL DIR     : /content/drive/MyDrive/SIPATURE/models/tfidf-aspect-silver-v1

REMINDER: output review-level (predictions/evidence) bersifat restricted.
Lanjut ke notebook 09 untuk prioritization + export (privacy-safe).
